# Agri Vision: EfficientDet FP32 speed benchmark

Open as rehmanfx71@gmail.com with T4 GPU. Keep the training search stopped. Run all. This performs short engineering measurements on training-only images, checks equivalent losses and parameter updates, and writes a separate benchmark report. It does not resume trials, save trained weights, or select hyperparameters.

Both loops remain full precision. Batch sizes 2/4 and resolutions 512/640 are tested three times with alternating order. Cached identical CPU batches isolate training-step overhead; data decoding, validation, FedAvg and Drive latency are excluded from the speed estimate.


In [ ]:
from pathlib import Path
DRIVE_ROOT = Path("/content/drive/MyDrive/Agri Vision")
WORKSPACE = Path("/content/agrivision_effdet_benchmark")
MODEL_FOLDER = DRIVE_ROOT / "grid_search/grid_v1_20261017/efficientdet_d0"
OUTPUT = DRIVE_ROOT / "benchmark_results/efficientdet_fp32_v1"
MODELS = ['efficientdet_d0']
WORKERS = 2
EXPECTED_DATASET_SHA256 = "7a2920e4a4e0405ee157e9b40ac6306b54d6933b20a306b44a4b7c6871f20868"
EXPECTED_CODE_SHA256 = "5453a56a97c2aca78224cec346c259e5f8a0a5eda2a93eb17b3677f06689c049"


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import hashlib, json, shutil, zipfile
assert DRIVE_ROOT.is_dir(), "Open this notebook as rehmanfx71@gmail.com; the copied Agri Vision folder must be in My Drive"
assert (DRIVE_ROOT / "71gmail_resume.json").is_file(), "Wrong Agri Vision folder: account migration marker missing"

def sha(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

WORKSPACE.mkdir(parents=True, exist_ok=True)
dataset_zip = WORKSPACE.parent / "non_chili_v1_colab_dataset.zip"
source_zip = DRIVE_ROOT / dataset_zip.name
if source_zip.is_file():
    if not dataset_zip.is_file() or sha(dataset_zip) != EXPECTED_DATASET_SHA256:
        shutil.copyfile(source_zip, dataset_zip)
else:
    if not dataset_zip.is_file() or sha(dataset_zip) != EXPECTED_DATASET_SHA256:
        parts_dir = DRIVE_ROOT / "non_chili_v1_upload_parts"
        manifest = json.loads((parts_dir / "parts.json").read_text())
        assert manifest["joined_sha256"] == EXPECTED_DATASET_SHA256, "Wrong dataset parts manifest"
        specs = manifest["parts"]
        assert specs and [p["name"] for p in specs] == [dataset_zip.name + f".part{i:02d}" for i in range(1, len(specs)+1)]
        partial = dataset_zip.with_suffix(".partial")
        with partial.open("wb") as output:
            for index, spec in enumerate(specs, 1):
                path = parts_dir / spec["name"]
                assert path.is_file() and path.stat().st_size == spec["bytes"], f"Missing/damaged part {index}"
                with path.open("rb") as stream:
                    shutil.copyfileobj(stream, output, 16*1024*1024)
                print(f"Dataset part {index}/{len(specs)} copied", flush=True)
        assert sha(partial) == EXPECTED_DATASET_SHA256, "Dataset parts damaged"
        partial.replace(dataset_zip)
assert sha(dataset_zip) == EXPECTED_DATASET_SHA256
dataset_marker = WORKSPACE / ".dataset_verified"
if not dataset_marker.exists() or dataset_marker.read_text() != EXPECTED_DATASET_SHA256:
    with zipfile.ZipFile(dataset_zip) as bundle:
        for name in bundle.namelist():
            assert (WORKSPACE/name).resolve().is_relative_to(WORKSPACE.resolve())
        bundle.extractall(WORKSPACE)
    dataset_marker.write_text(EXPECTED_DATASET_SHA256)
code_zip = WORKSPACE.parent / "agrivision_grid_search_v1_code.zip"
shutil.copyfile(DRIVE_ROOT/code_zip.name, code_zip)
assert sha(code_zip) == EXPECTED_CODE_SHA256, "Wrong grid-search code bundle"
with zipfile.ZipFile(code_zip) as bundle:
    for name in bundle.namelist():
        assert (WORKSPACE/name).resolve().is_relative_to(WORKSPACE.resolve())
    bundle.extractall(WORKSPACE)
print("Verified local dataset and isolated grid-search code.")


In [ ]:
import os, subprocess, sys
os.chdir(WORKSPACE)
sys.path.insert(0, str(WORKSPACE))
import torch, torchvision
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > T4 GPU"
assert "T4" in torch.cuda.get_device_name(0), "Use T4 for the comparison"
torch_before, vision_before = torch.__version__, torchvision.__version__
print("GPU:", torch.cuda.get_device_name(0), "torch:", torch_before, "torchvision:", vision_before)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-r", "requirements-grid.txt"], check=True)
# Preserve the existing CUDA packages while installing supporting libraries.
subprocess.run([sys.executable, "-m", "pip", "install", "huggingface_hub", "safetensors", "packaging", "tqdm", "omegaconf==2.4.0",
                "matplotlib", "opencv-python", "requests", "scipy", "psutil", "polars>=0.20", "ultralytics-thop>=2.0.18"], check=True)
assert torch.__version__ == torch_before and torchvision.__version__ == vision_before


In [ ]:
benchmark_zip = DRIVE_ROOT / "efficientdet_fp32_benchmark_v1.zip"
assert sha(benchmark_zip) == "99036645a92f6a530b3953675fde7ded3c29016ebc6e1da4896a01715a770b12", "Wrong benchmark bundle"
with zipfile.ZipFile(benchmark_zip) as bundle:
    for name in bundle.namelist():
        assert (WORKSPACE / name).resolve().is_relative_to(WORKSPACE.resolve())
    bundle.extractall(WORKSPACE)
command = [sys.executable, "-u", "-m", "benchmarks.efficientdet_fp32",
           "--root", str(WORKSPACE), "--model-folder", str(MODEL_FOLDER),
           "--output", str(OUTPUT), "--steps", "30", "--warmup", "3", "--repeats", "3"]
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1")) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()
if code:
    raise subprocess.CalledProcessError(code, command)
print("Benchmark complete. Report:", OUTPUT / "benchmark.json")
print("No search migration or full training was started.")


Review median speedups, memory and equivalence for every batch/resolution pair. Do not infer full-round savings directly from cached-batch measurements. No benefit, inconsistent timings or failed equivalence means do not migrate. A worthwhile result still requires an explicit reviewed continuation with versioned code, rather than editing stored hashes or trial identities.
